# Stage 6 — Evaluation and error analysis

**Goal.** Compare all models on the same test set (MAE, RMSE, ±k accuracy, band accuracy / macro-F1, R²), compute BERT's MAE gain with paired bootstrap CIs, analyse errors by band and occupation, and run the probe, OCR-noise, sensitivity and fairness analyses.

Logic: `src/evaluation.py`, `src/visualization.py`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import utils

utils.set_seed()
print("SEED:", utils.SEED, "| device:", utils.get_device(), "| root:", utils.ROOT)

All numbers come from `python -m src.evaluation`. It scores every model on the **same 600 test CVs, used here for the first time**, and writes the tables to `output/tables/`, the figures to `output/figures/` and a summary to `output/metrics/evaluation_summary.json`. Test predictions are cached in `output/predictions/`. Bootstraps use 2,000 paired resamples with seed 42.

The date rules are reported as an extra row: they are not a learned model, but they show the ceiling of the task.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

T = utils.TABLES_DIR
pd.set_option("display.max_colwidth", 160)

def show(name, w=16, h=5):
    fig, ax = plt.subplots(figsize=(w, h))
    ax.imshow(plt.imread(utils.FIGURES_DIR / name))
    ax.axis("off")
    ax.grid(False)
    plt.show()

## Model comparison on the test set

**In plain words.**
- **Median:** off by 3.5 years on average, with no information.
- **Frozen embeddings:** 1.26 years. They capture what a CV is about, but not how long.
- **TF-IDF:** 0.84 years. It reads year tokens as words.
- **Fine-tuned BERT:** 0.25 years (about 3 months). 98.5% of its predictions are within ±1 year, every prediction is within ±2 years, and R² is 0.994.
- **Date rules:** 0.02 years, with no training at all.

BERT costs about 17 minutes of training on the M2 against seconds for the baselines. At inference it takes about 8 ms per CV, similar to the embedding model.

In [ ]:
pd.read_csv(T / "model_comparison.csv")

## Question 1a · How much does BERT gain, and is it real?

Paired bootstrap: the same resampled test CVs are scored by both models 2,000 times, giving a 95% interval for the MAE reduction. **BERT's gain is statistically meaningful against every learned baseline:**
- vs median: −3.25 years [3.03, 3.47];
- vs TF-IDF: −0.59 years [0.53, 0.64], a 70% lower error;
- vs embeddings: −1.01 years [0.93, 1.09].

None of the intervals comes close to 0.

**But BERT is significantly worse than simple date arithmetic:** +0.23 years [0.21, 0.25]. In PopResume, `exp_year` is exactly the sum of the job durations, and a regex computes that exactly while BERT only approximates it.

In [ ]:
display(pd.read_csv(T / "bert_gain_bootstrap.csv"))
show("bert_gain.png", 10, 4.5)

## Predicted vs true: do models pull toward the middle?

**Why this chart.** Points on the dashed 45° line are perfect predictions. The orange fitted line shows the systematic pattern: a slope below 1 means juniors are overestimated and seniors underestimated, which is regression to the middle.

**What it shows.**
- **Embeddings** compress clearly: slope 0.86, seniors (11–20 years) underestimated by 1.35 years on average.
- **TF-IDF** compresses a little: slope 0.92, seniors −0.69.
- **BERT** does not compress: slope 0.99, and its small bias is a slight *overestimate* in every band (+0.2 years for juniors, +0.17 for seniors).
- **The rules** sit exactly on the line.

In [ ]:
show("pred_vs_true.png", 18, 4.5)
pd.read_csv(T / "compression_to_middle.csv")

## Question 1b · Where does BERT's gain come from?

**Why these charts.** Grouped bars put every model side by side in each experience band and each occupation, so you can see whether a gain is uniform or concentrated.

**What they show.**
- **Gain by band:** BERT's gain over TF-IDF is largest for **seniors** (1.16 → 0.29 years in the 11–20 band) and for juniors (0.78 → 0.30). Those are the extremes where the baselines are pulled toward the middle. Mid-career CVs gain too (0.80 → 0.19).
- **By occupation:** the picture is flat, with BERT between 0.22 and 0.29 years everywhere. The median baseline varies more by occupation (construction laborers have the most spread-out careers).
- **Bands:** BERT puts 92–98% of CVs in the right band (92% for 0–2 years). Its errors only spill into the neighbouring band, near the band edges.

In [ ]:
show("mae_by_band.png", 16, 6)
show("mae_by_occupation.png", 16, 6)
show("band_confusion.png", 18, 4)

## The 10 largest BERT errors

**All 10 are overestimates of about 1 to 1.6 years, and all 10 are career gaps.** The OCR read the dates correctly, nothing was truncated (no CV exceeds 162 tokens) and no jobs overlap. The rules get every one of these CVs right.

- **4 cases:** one short job (2–19 months) that ended around 2020, almost 3 years before the data snapshot (2023.01), with a much older graduation year. BERT predicts as if part of the time since then counted.
- **6 cases:** a gap of 0.8–2.5 years between two jobs. BERT predicts something between the true sum and the full first-start-to-last-end span.

**The pattern holds on the whole test set.**
- **Gap after the last job:** BERT's MAE doubles (0.43 vs 0.21) and it overestimates by +0.25 years.
- **Gaps between jobs:** BERT's MAE rises from 0.22 to 0.30.
- **TF-IDF** overestimates gap CVs even more (+0.50), because it reads the year range.
- **The rules** are unaffected.

**Possible improvements:**
- add the rules' output as an input feature, or train BERT on the residual of the rules;
- oversample CVs with gaps;
- try the Huber loss;
- give BERT normalised dates (e.g. months per job) instead of raw `YYYY.MM` strings.

In [ ]:
top = pd.read_csv(T / "bert_top10_errors.csv", dtype={"id": str})
display(top.drop(columns="text"))
display(pd.read_csv(T / "error_by_career_pattern.csv"))
print(top.loc[0, "text"])

## Question 1c · Probe: does the model read dates or just vocabulary?

20 test CVs with at least two jobs. Two edits, with the rules giving the change a recruiter would expect:

- **(a) Delete one job.** The expected change is minus its duration: **−2.65 years** on average.
  - **BERT** moves in the right direction in 20/20 CVs, but only by −1.79 years.
  - **TF-IDF** moves by −1.15 years and **embeddings** by −1.72.
  - The correlation between each learned model's change and the deleted job's actual length is near 0 (BERT −0.07). They react to *one job fewer*, not to *how long that job was*.
- **(b) Shift every date by +5 years.** No CV says "Present", so durations do not change and the expected change is **0**.
  - **BERT** drops by **−1.45 years** on average.
  - **TF-IDF** and **embeddings** drop by about −0.55.
  - The rules do not move.

**Conclusion.** BERT reads dates much better than the baselines on normal CVs (0.25 vs 0.84 MAE), but the probe shows it has *not* learned the arithmetic.
- Part of what it knows is tied to the **absolute years**: later years mean less experience, because the data ends in 2023.01, so dates near 2023 signal a recent start.
- The rest is tied to the **number of job lines**.
- Shifting the dates into the future is out of distribution, and BERT fails there in a way a recruiter would not. Only the rules read dates the way a recruiter does.

In [ ]:
display(pd.read_csv(T / "q1_probe_summary.csv"))
pd.read_csv(T / "q1_probe_predictions.csv", dtype={"id": str}).head()

## Question 2 · How much does OCR noise cost?

**OCR quality (stage 3):**
- CER 0.21% and WER 1.9%.
- The one systematic failure is the dropped dash in date ranges, so only 79.8% of years are found as is. After `repair_date_ranges`, 99.996% are found.
- The remaining errors are `i`→`l` in acronyms and dates cut off by the CV rendering itself.

**Cost for the models.** Every clean-trained model is scored on its test CVs twice: on the cleaned OCR text and on the cleaned reference text.
- **The gaps are tiny:** under 0.015 years for every model, and not significant for the baselines.
- **BERT is 0.007 years *better* on OCR text** than on the reference. That is significant but negligible. It was trained on OCR text, so the reference text is slightly out of its training distribution (for example, full `2023.01` dates where the images show `2023.`).
- **Most robust:** after the date repair, OCR noise costs essentially nothing. The bag-of-words and embedding models hardly care about single characters, and BERT and the rules see the same dates either way.

In [ ]:
pd.read_csv(T / "q2_ocr_vs_reference.csv")

## Question 3 · Methodological sensitivity: raw vs cleaned OCR

Each model is trained and tested on each text variant:
- `raw`: no repair, so dates are still glued;
- `clean`;
- `clean_dropped_lowconf`: lines below 0.97 confidence removed. BERT was not trained on this variant.

**Results on test:**
- **BERT is the model most sensitive to cleaning:** 0.42 with raw vs 0.25 with clean (+0.17 years, paired bootstrap CI [0.14, 0.20]). On raw text it sees `2020.112023.01` as one long token sequence, and the arithmetic gets harder.
- **Embeddings** lose 0.08 years with raw (significant). **TF-IDF** barely changes (+0.01, not significant), because its character n-grams still catch the glued dates.
- **Dropping low-confidence lines** hurts slightly or not at all (TF-IDF +0.02, rules +0.015), as predicted in stage 3: those lines are correct, and 36 of them carry job dates.

**Is BERT's raw-vs-clean gap real or noise?** The paired bootstrap only covers *test-set* noise, and on that the gap is clearly significant. It does not cover *training* noise: each variant was trained once, and the validation MAE moved by 0.05–0.2 years between consecutive epochs (raw: 0.40 → 0.61 → 0.46). The raw run also stopped at epoch 2, while clean stopped at epoch 4. So the direction is credible (cleaning helps, and the mechanism is clear), but the size of the gap should be confirmed by retraining both variants with 2–3 seeds before it is stated as a fact.

In [ ]:
display(pd.read_csv(T / "q3_variant_mae.csv"))
pd.read_csv(T / "q3_variant_bootstrap.csv")

## Question 4 · Fairness check (sex and race are not model inputs)

**BERT's mean error (predicted − true) by group, with 95% bootstrap CIs.** BERT slightly overestimates every group except Black candidates:

| Group | n | Mean error (years) | 95% CI |
|---|---|---|---|
| Female | 299 | +0.08 | [0.05, 0.13] |
| Male | 301 | +0.14 | [0.10, 0.17] |
| Asian/Pacific Islander | 91 | +0.15 | [0.09, 0.22] |
| White | 462 | +0.11 | [0.09, 0.14] |
| Black | 47 | −0.02 | [−0.12, 0.07] |

- **MAE is the same across all groups (0.24–0.26).**
- The sex intervals overlap. The Asian/PI vs Black difference is about 0.18 years (2 months), with only 47 Black candidates in the test set.
- **The date rules show no group difference at all** (mean errors within ±0.004 years).

**What could explain the differences when sex and race are not inputs?** Group membership correlates with career patterns in this population-based data, and BERT's bias depends on those patterns. Its largest overestimates are careers with a gap after the last job, and men have that pattern more often (19.9% vs 16.7%). The groups also differ in average experience (Asian/PI 7.5 years vs White 6.3). Names, which could reveal sex or race, are removed before modelling. The differences are small compared with the overall error, but **a model can pick up group-correlated proxies even without the attributes**, so this check must be repeated on real data.

In [ ]:
display(pd.read_csv(T / "q4_fairness.csv"))
show("q4_fairness.png", 10, 4.5)

## Question 4 · Business value and limits

- **Who could use it.** Recruiting agencies, job boards, public employment services and HR teams. The right task is **pre-filling a profile field** ("years of experience: ~6") that a person confirms, not **deciding who gets an interview**.
- **What an error costs.** ±2 years rarely matters for a profile field. It matters at thresholds ("5+ years required"): a 4.5-year candidate predicted at 5.5 gets through, and the reverse silently filters someone out. Underestimating is worse for the candidate (hidden rejection); overestimating is worse for the employer (mismatched interviews). A screening use should therefore always show the extracted dates, not just a number.
- **Is BERT worth it?** Against the learned baselines, clearly yes: 0.25 vs 0.84 years MAE, a significant 70% reduction, with no compression toward the middle, for about 17 minutes of training on a laptop and 8 ms per CV. **Against date arithmetic, no:** on this data the rules are 10× more accurate, instantaneous and fully explainable. The probe shows BERT has not truly learned to read dates. The best system would combine both: rules for well-formatted CVs, a learned model for messy text where the regex finds no clean date ranges.
- **Limitations:**
  - synthetic resumes from a single template, all one column, with the target defined exactly by the listed dates;
  - only 5 occupations and U.S. statistics, English only;
  - one training run per variant.

  Real CVs have free-form dates ("since 2019", "3 yrs"), overlapping and part-time jobs, and multi-column layouts, so these numbers are an upper bound. **Such a system must never decide on its own who is hired.**